# Forecasting WPUSI01102B - Split fijo, transformaciones y manejo de anomalia

Experimento con **split temporal fijo** (sin cross-validation):

- **Train:** hasta 2021-12   |   **Test:** desde 2022-01
- **Periodo COVID** marcado/tratado: 2020-03 a 2020-10

Partes del experimento:
- **A:** modelos sklearn x transformaciones x tamanos de ventana
- **B:** 6 estrategias de manejo del shock COVID (none, dummy, winsorize, exclude, isolation, prophet)
- **C:** modelos clasicos (Holt-Winters, SARIMA, Theta, SeasonalNaive)
- **D:** una red neuronal (MLP)
- **E:** efecto del TDA (mismo modelo con vs sin caracteristicas topologicas)

In [ ]:
# Silenciar warnings (ANTES de importar tensorflow/prophet)
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import warnings; warnings.filterwarnings("ignore")
import logging
for _l in ["tensorflow", "prophet", "cmdstanpy"]:
    logging.getLogger(_l).setLevel(logging.ERROR)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from berry_price_tda.data.loader import (
    load_dataset, date_train_test_split, TARGET, TRAIN_END, TEST_START,
)
from berry_price_tda.features.anomaly import covid_mask, COVID_START, COVID_END
from berry_price_tda.pipelines.experiment import run_full_experiment

print("Split -> train hasta", TRAIN_END, "| test desde", TEST_START)

## 1. Datos y marcado del periodo anomalo

In [ ]:
df = load_dataset("../../data/interim/berry_features.csv")
train, test = date_train_test_split(df)
print(f"Train: {len(train)} obs ({train.index.min().date()} -> {train.index.max().date()})")
print(f"Test : {len(test)} obs ({test.index.min().date()} -> {test.index.max().date()})")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df[TARGET], 'b-', lw=1.5, label="WPUSI01102B")
ax.axvspan(pd.Timestamp(COVID_START), pd.Timestamp(COVID_END),
           color='red', alpha=0.2, label="COVID")
ax.axvline(pd.Timestamp(TEST_START), color='green', ls='--', label="Inicio test")
ax.legend(); ax.grid(alpha=0.3); ax.set_title("Serie con periodo COVID y split")
plt.tight_layout(); plt.show()

## 2. Experimento completo

In [ ]:
resultados = run_full_experiment(
    data_path="../../data/interim/berry_features.csv",
    windows=[6, 12, 18, 24],
    transforms=["none", "standard", "diff", "log_diff", "seasonal_diff"],
    anomaly_window=12,
    anomaly_transform="diff",
    run_tda=True,          # Parte E: con vs sin TDA
    tda_window=24,         # el TDA necesita ventana amplia (mas puntos en la nube)
    tda_transform="none",  # el embedding de Takens opera sobre el nivel
    sort_by="mae",
    run_nn=True,           # False para corrida mas rapida
    verbose=True,
)
resultados.head(15)

## 3. Comparacion por parte

In [ ]:
# Mejor de cada parte
print("Mejor combinacion por parte (menor MAE):")
for parte in resultados["parte"].unique():
    best = resultados[resultados["parte"]==parte].sort_values("mae").iloc[0]
    print(f"  {parte:10s}: {best['modelo']:18s} MAE={best['mae']:.3f} R2={best['r2']:.3f}")

### Efecto de las estrategias anti-COVID

In [ ]:
anom = resultados[resultados["parte"]=="anomaly"]
efecto = anom.groupby("estrategia")[["mae","rmse","r2"]].mean().sort_values("mae")
print("MAE promedio por estrategia de manejo del shock:")
display(efecto)

fig, ax = plt.subplots(figsize=(9,4))
efecto["mae"].plot(kind="barh", ax=ax, color="#D85A30")
ax.set_xlabel("MAE promedio"); ax.set_title("Estrategias anti-COVID")
ax.grid(axis="x", alpha=0.3); plt.tight_layout(); plt.show()

### Efecto de las transformaciones

In [ ]:
tf_eff = resultados[resultados["parte"]=="transform"]
piv = tf_eff.groupby("transform")["mae"].mean().sort_values()
print("MAE promedio por transformacion:")
display(piv)

fig, ax = plt.subplots(figsize=(9,4))
piv.plot(kind="barh", ax=ax, color="#378ADD")
ax.set_xlabel("MAE promedio"); ax.set_title("Transformaciones de la serie")
ax.grid(axis="x", alpha=0.3); plt.tight_layout(); plt.show()

### Efecto del TDA (con vs sin caracteristicas topologicas)

Compara el mismo modelo con y sin features topologicas. El TDA se calcula
sobre la ventana en NIVEL (serie original), respetando la teoria de Takens.

In [ ]:
tda = resultados[resultados["parte"]=="tda"]
if not tda.empty and "use_tda" in tda.columns:
    piv = tda.pivot_table(index="modelo", columns="use_tda", values="mae")
    piv.columns = ["sin_TDA", "con_TDA"]
    piv["delta_MAE"] = piv["con_TDA"] - piv["sin_TDA"]
    print("Efecto del TDA sobre el MAE (positivo = empeora, negativo = mejora):")
    display(piv.round(3))

    fig, ax = plt.subplots(figsize=(9,4))
    piv[["sin_TDA","con_TDA"]].plot(kind="bar", ax=ax,
                                     color=["#888780","#534AB7"])
    ax.set_ylabel("MAE"); ax.set_title("MAE con vs sin TDA por modelo")
    ax.grid(axis="y", alpha=0.3); plt.xticks(rotation=0)
    plt.tight_layout(); plt.show()
else:
    print("No hay resultados de TDA (giotto-tda no disponible o run_tda=False)")

## 4. Top global y guardado

In [ ]:
cols = ["parte","modelo","transform","window","estrategia","mae","rmse","r2"]
cols = [c for c in cols if c in resultados.columns]
print("Top 10 global (por MAE):")
display(resultados[cols].head(10))

resultados.to_csv("../../data/processed/comparacion_modelos.csv", index=False)
print("Guardado en data/processed/comparacion_modelos.csv")